# Fraudulent Transaction Detection Using DBSCAN

## DBSCAN-Based Unsupervised Anomaly Detection

This notebook builds an unsupervised fraud/anomaly detection pipeline using **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**.

### Workflow
1. Load and validate the transaction dataset
2. Prepare features for unsupervised learning
3. Engineer useful behavioral/time features
4. Encode categorical variables and scale numeric variables
5. Select DBSCAN parameters using a k-distance graph
6. Detect anomalies using DBSCAN noise points (`-1`)
7. Evaluate anomalies against the known fraud labels
8. Visualize clusters and anomalies
9. Compare parameter choices
10. Export final results

> **Important:** `Fraud_Label` is NOT used as an input to DBSCAN. It is used only for post-hoc evaluation.

## 1. Import Libraries & Load Dataset

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

In [ ]:
# Upload the CSV when running in Google Colab
from google.colab import files

uploaded = files.upload()
csv_file = next(name for name in uploaded if name.lower().endswith(".csv"))
df = pd.read_csv(csv_file)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 2. Data Inspection & Validation

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nTotal Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
print("Fraud Label Distribution:")
print(df["Fraud_Label"].value_counts().sort_index())

print("\nFraud Percentage:")
print((df["Fraud_Label"].value_counts(normalize=True).sort_index() * 100).round(2))

plt.figure(figsize=(7,5))
sns.countplot(data=df, x="Fraud_Label")
plt.title("Fraud vs Non-Fraud Transactions")
plt.xlabel("Fraud Label (0 = Normal, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.show()

## 3. Prepare Data for Unsupervised Learning

`Fraud_Label` is saved separately for evaluation only. Identifiers are removed because they do not represent transaction behavior.

In [ ]:
y_true = df["Fraud_Label"].copy()

X = df.drop(
    columns=["Fraud_Label", "Transaction_ID", "Customer_ID"]
).copy()

print("Fraud_Label excluded from DBSCAN input.")
print("Identifiers excluded from DBSCAN input.")
print("\nFeatures used for DBSCAN:")
print(X.columns.tolist())

## 4. Feature Engineering

Transaction hour is represented cyclically so that times near midnight remain close to each other. Selected highly skewed numeric variables are log-transformed before scaling.

In [ ]:
X["Hour_Sin"] = np.sin(2 * np.pi * X["Transaction_Hour"] / 24)
X["Hour_Cos"] = np.cos(2 * np.pi * X["Transaction_Hour"] / 24)
X.drop(columns=["Transaction_Hour"], inplace=True)

log_features = [
    "Transaction_Amount",
    "Location_Distance_KM",
    "Login_Attempts",
    "Account_Balance",
    "Amount_Deviation",
    "Previous_Transaction_Avg"
]

for col in log_features:
    X[col] = np.log1p(X[col])

print("Feature engineering completed.")
print("Current shape:", X.shape)

## 5. Encode & Scale Features

DBSCAN is distance-based, so categorical variables are one-hot encoded and numeric variables are transformed using `RobustScaler`.

In [ ]:
categorical_features = ["Location", "Transaction_Type", "Channel"]

numerical_features = [
    "Transaction_Amount",
    "Transaction_Frequency_24H",
    "Location_Distance_KM",
    "Device_Change",
    "International_Transaction",
    "Login_Attempts",
    "Account_Age_Days",
    "Previous_Transaction_Avg",
    "Amount_Deviation",
    "Merchant_Risk_Score",
    "Transaction_Duration_Sec",
    "Account_Balance",
    "Amount_to_Balance_Ratio",
    "Hour_Sin",
    "Hour_Cos"
]

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("num", RobustScaler(), numerical_features),
    ("cat", encoder, categorical_features)
])

X_scaled = preprocessor.fit_transform(X)

print("Preprocessing completed.")
print("Rows:", X_scaled.shape[0])
print("Processed features:", X_scaled.shape[1])

## 6. Select DBSCAN Parameters with a K-Distance Graph

DBSCAN requires `eps` and `min_samples`.

We use the k-distance graph to identify a sensible neighborhood-radius range from the structure of the feature space. For this dataset, the elbow region is around **2.9**, so `eps = 2.9` is selected for the final model.

> **Important:** `Fraud_Label` is not used to choose `eps`. The fraud labels are used only later for post-hoc evaluation and sensitivity analysis.

In [ ]:
min_samples = 5

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_scaled)

distances, _ = neighbors.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.title(f"{min_samples}-Nearest Neighbor Distance Graph")
plt.xlabel("Transactions sorted by distance")
plt.ylabel(f"Distance to {min_samples}th Neighbor")
plt.grid(True)
plt.show()

print("The elbow / sharp rise is inspected to guide eps selection.")
print("For this dataset, the selected starting value is eps = 2.9.")


## 7. Run DBSCAN & Inspect Clusters

DBSCAN assigns `-1` to observations considered noise. In this project, those noise observations are treated as candidate anomalies.

In [ ]:
# Selected from the k-distance graph
eps = 2.9

dbscan = DBSCAN(
    eps=eps,
    min_samples=min_samples
)

cluster_labels = dbscan.fit_predict(X_scaled)

print("DBSCAN completed.")
print("eps =", eps)
print("min_samples =", min_samples)


In [ ]:
unique_labels, counts = np.unique(cluster_labels, return_counts=True)
cluster_summary = pd.DataFrame({
    "Cluster": unique_labels,
    "Transactions": counts
}).sort_values("Cluster")

display(cluster_summary)

num_anomalies = int((cluster_labels == -1).sum())
print("Detected DBSCAN anomalies:", num_anomalies)
print("Anomaly percentage:", round(num_anomalies / len(cluster_labels) * 100, 2), "%")

## 8. Create the Anomaly Results Table

In [ ]:
results_df = df.copy()
results_df["DBSCAN_Cluster"] = cluster_labels
results_df["DBSCAN_Anomaly"] = (results_df["DBSCAN_Cluster"] == -1).astype(int)

print("Results table created.")
display(results_df.head())

## 9. Evaluate Anomalies Against Known Fraud Labels

The fraud labels are used only after DBSCAN has completed. This is a post-hoc evaluation of how well the unsupervised anomalies correspond to the known labels.

In [ ]:
y_pred = results_df["DBSCAN_Anomaly"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("DBSCAN Anomaly Detection Performance")
print("=" * 45)
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1-Score : {f1 * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Normal", "Fraud"], zero_division=0))

## 10. Confusion Matrix & PCA Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Fraud"],
            yticklabels=["Normal", "Fraud"])
plt.title("DBSCAN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# PCA is used only to visualize the high-dimensional feature space in 2D.
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("PCA explained variance:", round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

clustered = cluster_labels != -1
anomalies = cluster_labels == -1

plt.figure(figsize=(10,7))
plt.scatter(X_pca[clustered, 0], X_pca[clustered, 1], s=30, alpha=0.5, label="Clustered")
plt.scatter(X_pca[anomalies, 0], X_pca[anomalies, 1], s=70, marker="X", label="DBSCAN Anomaly")
plt.title("DBSCAN-Based Anomaly Detection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.show()

## 11. Compare DBSCAN Parameter Choices

We test several `eps` values while keeping `min_samples` fixed.

This is a **sensitivity analysis** showing how the model behaves across nearby parameter values.

The final `eps` is **not selected by maximizing the fraud-label metrics** here. Instead, the k-distance graph provides the primary basis for choosing the final `eps`; the known fraud labels are used only to evaluate how the resulting anomalies align with the dataset labels.

In [ ]:
parameter_results = []

for eps_value in np.arange(2.5, 3.31, 0.1):
    current_eps = round(float(eps_value), 2)

    model = DBSCAN(
        eps=current_eps,
        min_samples=min_samples
    )

    labels = model.fit_predict(X_scaled)
    predictions = (labels == -1).astype(int)

    parameter_results.append({
        "eps": current_eps,
        "clusters": len(set(labels)) - (1 if -1 in labels else 0),
        "anomalies": int(predictions.sum()),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1_score": f1_score(
            y_true, predictions, zero_division=0
        )
    })

parameter_results = pd.DataFrame(parameter_results)
display(parameter_results.round(4))

print("Sensitivity analysis complete.")
print("Final eps remains 2.9 based on the k-distance graph.")


In [ ]:
plt.figure(figsize=(10,6))
plt.plot(parameter_results["eps"], parameter_results["precision"], marker="o", label="Precision")
plt.plot(parameter_results["eps"], parameter_results["recall"], marker="o", label="Recall")
plt.plot(parameter_results["eps"], parameter_results["f1_score"], marker="o", label="F1-Score")
plt.xlabel("eps")
plt.ylabel("Score")
plt.title("DBSCAN Performance Across eps Values")
plt.legend()
plt.grid(True)
plt.show()

## 12. Final Model & Suspicious Transaction Report

The final model uses the `eps` value selected from the k-distance analysis (`2.9`) and the chosen `min_samples` value (`5`).

In [ ]:
FINAL_EPS = 2.9
FINAL_MIN_SAMPLES = 5

final_dbscan = DBSCAN(
    eps=FINAL_EPS,
    min_samples=FINAL_MIN_SAMPLES
)

final_labels = final_dbscan.fit_predict(X_scaled)

results_df["Final_DBSCAN_Cluster"] = final_labels
results_df["Final_Anomaly"] = (
    final_labels == -1
).astype(int)

final_pred = results_df["Final_Anomaly"]

final_accuracy = accuracy_score(y_true, final_pred)
final_precision = precision_score(
    y_true, final_pred, zero_division=0
)
final_recall = recall_score(
    y_true, final_pred, zero_division=0
)
final_f1 = f1_score(
    y_true, final_pred, zero_division=0
)

print("=" * 55)
print("FINAL DBSCAN RESULTS")
print("=" * 55)
print(f"Parameter basis: k-distance graph")
print(f"eps         : {FINAL_EPS}")
print(f"min_samples : {FINAL_MIN_SAMPLES}")
print(f"Anomalies   : {int(final_pred.sum())}")
print(f"Accuracy    : {final_accuracy * 100:.2f}%")
print(f"Precision   : {final_precision * 100:.2f}%")
print(f"Recall      : {final_recall * 100:.2f}%")
print(f"F1-Score    : {final_f1 * 100:.2f}%")


In [ ]:
suspicious_transactions = results_df[results_df["Final_Anomaly"] == 1].copy()

columns_to_show = [
    "Transaction_ID", "Customer_ID", "Transaction_Amount",
    "Transaction_Hour", "Transaction_Frequency_24H", "Location",
    "Location_Distance_KM", "Device_Change",
    "International_Transaction", "Login_Attempts",
    "Merchant_Risk_Score", "Amount_Deviation",
    "Amount_to_Balance_Ratio", "Fraud_Label",
    "Final_DBSCAN_Cluster"
]

print("Suspicious transactions detected:", len(suspicious_transactions))
display(suspicious_transactions[columns_to_show].head(20))

## 13. Export Final Results

In [ ]:
results_df.to_csv("DBSCAN_Final_Results.csv", index=False)
suspicious_transactions.to_csv("DBSCAN_Suspicious_Transactions.csv", index=False)

print("Files created successfully:")
print("- DBSCAN_Final_Results.csv")
print("- DBSCAN_Suspicious_Transactions.csv")

## 14. Conclusion

The project uses DBSCAN as an **unsupervised anomaly detection** technique for fraudulent transaction analysis.

### Key steps completed
- Validated the transaction dataset
- Excluded fraud labels from DBSCAN training
- Removed identifier columns
- Engineered cyclical time features
- Reduced skew in selected numeric features
- One-hot encoded categorical features
- Robust-scaled numeric features
- Used a k-distance graph to guide `eps` selection
- Applied DBSCAN clustering
- Treated DBSCAN noise (`-1`) as candidate anomalies
- Used known fraud labels only for post-hoc evaluation and sensitivity analysis
- Visualized the anomaly structure using PCA
- Exported the final results

### Final model interpretation
The final `eps` is selected from the k-distance graph rather than by maximizing fraud-label performance. The fraud labels are then used only to evaluate how closely the unsupervised anomalies align with the known fraud cases.

A DBSCAN anomaly is **not automatically proof of fraud**. It means the transaction is unusual relative to the learned density structure. The comparison with `Fraud_Label` tells us how closely these unsupervised anomalies align with the known labels in this dataset.
